In [1]:
import torch
import json
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
import os

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


# LOAD Model

In [3]:
model_name = "Qwen/Qwen3-8B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    torch_dtype="auto",
    device_map = "auto"
)


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [4]:
print(model.device)

cuda:0


In [5]:
# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

thinking content: 
content: A large language model (LLM) is a type of artificial intelligence that is trained on vast amounts of text data to understand and generate human-like text. These models can perform a wide range of tasks, such as answering questions, writing stories, coding, and translating languages. LLMs are called "large" because they have a massive number of parameters, which allows them to capture complex patterns in language and produce highly accurate and contextually relevant responses. They are widely used in various applications, including chatbots, virtual assistants, content creation, and more.


# FEW SHOT TEMPLATES

In [6]:
# Few-shot template 
FEW_SHOT_HINDI = """
# Instruction:
You are a helpful assistant who generates answers from a Hindi table to answer Hindi questions.  
Use the below example to guide the format. 

## Example:

### Input:
 ले ट्रुनिन ने किन फिल्मों में भूमिका निभाई थी?  

<column>  वर्ष | शीर्षक | भूमिका  
<row 1> 2014 | See No Evil 2 | जैकब गुडनाइट  
<row 2> 2016 | Countdown | ले ट्रुनिन  
<row 3> 2017 | Meltdown | ले ट्रुनिन  

### Response (complete this):
<column> शीर्षक  
<row 1> Countdown  
<row 2> Meltdown  

now answer the following question, generate only the ouput table and nothing else.
"""

FEW_SHOT_TELUGU = """
# Instruction:
You are a helpful assistant who generates answers from a Telugu table to answer Telugu questions.  
Use the below example to guide the format. 

## Example:

### Input:
 లే ట్రునిన్ ఏ సినిమాలలో పాత్ర పోషించాడు?

<column>  సంవత్సరం | శీర్షిక | పాత్ర  
<row 1> 2014 | See No Evil 2 | జేకబ్ గుడ్‌నైట్  
<row 2> 2016 | Countdown | లే ట్రునిన్  
<row 3> 2017 | Meltdown | లే ట్రునిన్  

### Response (complete this):
<column> శీర్షిక  
<row 1> Countdown  
<row 2> Meltdown  

now answer the following question, generate only the ouput table and nothing else.
"""

FEW_SHOT_BENGALI = """
# Instruction:
You are a helpful assistant who generates answers from a Bengali table to answer Bengali questions.  
Use the below example to guide the format. 

## Example:

### Input:
 লে ট্রুনিন কোন কোন সিনেমায় অভিনয় করেছেন?

<column>  বছর | শিরোনাম | চরিত্র  
<row 1> 2014 | See No Evil 2 | জ্যাকব গুডনাইট  
<row 2> 2016 | Countdown | লে ট্রুনিন  
<row 3> 2017 | Meltdown | লে ট্রুনিন  

### Response (complete this):
<column> শিরোনাম  
<row 1> Countdown  
<row 2> Meltdown  

now answer the following question, generate only the ouput table and nothing else.
"""


In [7]:
# Load test.json
with open("test.json", "r") as f:
    test_data = json.load(f)



def table_to_text(table):
    if not table:
        return ""
    columns = list(table[0].keys())
    header = " | ".join(columns)
    rows = []
    for i, row in enumerate(table):
        row_str = " | ".join(str(row[col]) for col in columns)
        rows.append(f"<row {i+1}> {row_str}")
    return f"<column>\n{header}\n" + "\n".join(rows)

def resultdf_to_text(result_df):
    if not result_df:
        return ""
    df = pd.DataFrame(result_df)
    return df.to_string(index=False)



In [8]:
# # ...existing code...
# results = []

# total_queries = sum(len(table["queries"]) for table in test_data)

# with tqdm(total=total_queries, desc="Processing queries") as pbar:
#     for table in test_data:
#         table_name = table["table_name"]
#         full_table = table["full_table_df"]
#         table_text = table_to_text(full_table)
#         for q in table["queries"]:
#             question = q["question"]
#             result_df = q.get("result_df", [])
#             # Compose the prompt
#             prompt = FEW_SHOT + f"\n###Input:\n{question}\n{table_text}\n###Response:\n"
#             # Prepare chat template for Sarvam
#             messages = [{"role": "user", "content": prompt}]
#             text = tokenizer.apply_chat_template(
#                 messages,
#                 tokenize=False,
#             )
#             model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
#             generated_ids = model.generate(**model_inputs, max_new_tokens=1024)
#             output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
#             output_text = tokenizer.decode(output_ids)
#             # Extract only the content after ###Response:
#             if "###Response:" in output_text:
#                 model_response = output_text.split("###Response:")[-1].strip()
#             else:
#                 model_response = output_text.strip()

#             # Store all relevant outputs in a dict
#             result_entry = {
#                 "question": question,
#                 "table_name": table_name,
#                 "table_text": table_text,
#                 "model_response": model_response,
#                 "actual_result_df": resultdf_to_text(result_df),
#                 "raw_model_output": output_text,
#             }
#             results.append(result_entry)

#             pbar.update(1)

# # Save results as JSON
# with open("qwen3_no_think.json", "w", encoding="utf-8") as f:
#     json.dump(results, f, ensure_ascii=False, indent=2)
# # ...existing code...

# `run_tableqa_testset()`

In [9]:
def run_tableqa_testset(model, tokenizer, model_instruction, testset_path, result_path, sanity=False, thinking_mode=True):
    """
    Loads a test set from testset_path, prompts the model for each query, and writes results to result_path.
    If sanity=True, processes only one question and saves with sanity_ prefix.
    """
    generation_config = {
        "temperature": 0.6,
        "top_p": 0.95,
        "top_k": 20,
        "min_p": 0.0,
        # "do_sample": True,
        "max_new_tokens": 1024
    }
    with open(testset_path, "r", encoding="utf-8") as f:
        test_data = json.load(f)

    tables_to_process = test_data if not sanity else [test_data[0]]
    total_queries = 1 if sanity else sum(len(table["queries"]) for table in tables_to_process)

    with tqdm(total=total_queries, desc="Processing queries") as pbar:
        for table in tables_to_process:
            full_table = table["full_table_df"]
            table_text = table_to_text(full_table)
            queries_to_process = [table["queries"][0]] if sanity else table["queries"]
            
            for q in queries_to_process:
                question = q["question"]
                prompt = f"\n###Input:\n{question}\n{table_text}\n###Response:\n"
                messages = [
                    {"role": "system", "content": model_instruction},
                    {"role": "user", "content": prompt}
                ]
                text = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=thinking_mode # Switches between thinking and non-thinking modes. Default is True.
                )
                model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
                generated_ids = model.generate(**model_inputs, **generation_config)
                output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
                output_text = tokenizer.decode(output_ids)

                # Try to split into thinking content and response content
                try:
                    # Look for </think> token
                    index = len(output_ids) - output_ids[::-1].index(151668)  # 151668 is </think> token
                    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
                    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
                except ValueError:
                    # If </think> not found, put everything in content
                    thinking_content = ""
                    content = output_text.strip()

                # Extract response after ###Response: if present
                if "###Response:" in content:
                    content = content.split("###Response:")[-1].strip()

                q["model_response"] = {
                    "thinking_content": thinking_content,
                    "content": content
                }

                pbar.update(1)
                if sanity:
                    print("prompt:\n", prompt)
                    print("Thinking content:", thinking_content)
                    print("Response content:", content)
                    
                    # Save with sanity_ prefix for sanity check
                    dir_path = os.path.dirname(result_path)
                    base_name = os.path.basename(result_path)
                    sanity_path = os.path.join(dir_path, "sanity_" + base_name)
                    with open(sanity_path, "w", encoding="utf-8") as f:
                        json.dump([table], f, ensure_ascii=False, indent=2)
                    return

    if not sanity:
        with open(result_path, "w", encoding="utf-8") as f:
            json.dump(test_data, f, ensure_ascii=False, indent=2)

In [12]:
run_tableqa_testset(
    model=model,
    tokenizer=tokenizer, 
    model_instruction=FEW_SHOT_BENGALI, 
    testset_path="data/bengali/bengali_testset.json", 
    result_path="data/bengali/qwen3_8B_results_nothink.json", 
    # sanity=True,
    thinking_mode=False
)

Processing queries:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing queries: 100%|██████████| 1000/1000 [1:22:28<00:00,  4.95s/it] 


In [ ]:
run_tableqa_testset(
    model=model,
    tokenizer=tokenizer, 
    model_instruction=FEW_SHOT_BENGALI, 
    testset_path="data/bengali/bengali_testset.json", 
    result_path="data/bengali/qwen3_8B_results.json", 
    # sanity=True,
    thinking_mode=True
)

Processing queries: 100%|██████████| 1000/1000 [6:58:52<00:00, 25.13s/it] 


: 